# Proyecto 1 - Obtención y Limpieza de Datos

## Universidad del Valle de Guatemala

### CC3084 - Data Science

---

## Objetivo

Automatizar la obtención de los establecimientos educativos del Ministerio de Educación de Guatemala correspondientes al nivel **Diversificado** para todos los departamentos del país.

Todo el proceso debe ser reproducible mediante código.

---

Autor: Osman Emanuel de León García
```

In [11]:
# Instalación de librerías

%pip install selenium webdriver-manager pandas openpyxl xlrd

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import os
import time
from pathlib import Path

import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from selenium.common.exceptions import TimeoutException
from selenium.common.exceptions import NoSuchElementException

from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service

In [13]:
# -----------------------------
# Configuración del proyecto
# -----------------------------

from pathlib import Path

URL = "https://www.mineduc.gob.gt/BUSCAESTABLECIMIENTO_GE/wbfBuscar.aspx"

# Estructura de carpetas
DATA_FOLDER = Path("data")

RAW_FOLDER = DATA_FOLDER / "raw"
CSV_FOLDER = DATA_FOLDER / "csv"
CLEAN_FOLDER = DATA_FOLDER / "clean"

RAW_FOLDER.mkdir(parents=True, exist_ok=True)
CSV_FOLDER.mkdir(parents=True, exist_ok=True)
CLEAN_FOLDER.mkdir(parents=True, exist_ok=True)

DOWNLOAD_TIMEOUT = 60

In [14]:
departamentos = {
    "16": "Alta_Verapaz",
    "15": "Baja_Verapaz",
    "04": "Chimaltenango",
    "20": "Chiquimula",
    "00": "Ciudad_Capital",
    "02": "El_Progreso",
    "05": "Escuintla",
    "01": "Guatemala",
    "13": "Huehuetenango",
    "18": "Izabal",
    "21": "Jalapa",
    "22": "Jutiapa",
    "17": "Peten",
    "09": "Quetzaltenango",
    "14": "Quiche",
    "11": "Retalhuleu",
    "03": "Sacatepequez",
    "12": "San_Marcos",
    "06": "Santa_Rosa",
    "07": "Solola",
    "10": "Suchitepequez",
    "08": "Totonicapan",
    "19": "Zacapa"
}

In [15]:
def esperar_elemento(driver, by, value, timeout=20):
    """
    Espera hasta que un elemento esté presente.
    """
    return WebDriverWait(driver, timeout).until(
        EC.presence_of_element_located((by, value))
    )


def esperar_click(driver, by, value, timeout=20):
    """
    Espera hasta que un elemento sea clickeable.
    """
    return WebDriverWait(driver, timeout).until(
        EC.element_to_be_clickable((by, value))
    )


def esperar_descarga(carpeta, timeout=60):

    tiempo = time.time()

    while time.time() - tiempo < timeout:

        archivos_tmp = list(Path(carpeta).glob("*.crdownload"))

        if not archivos_tmp:

            archivo = Path(carpeta) / "establecimiento.xls"

            if archivo.exists():
                return archivo

        time.sleep(1)

    raise TimeoutException("La descarga tardó demasiado.")

In [16]:
def renombrar_excel(nombre_departamento):
    """
    Renombra el archivo descargado con el nombre del departamento.
    """

    archivo = RAW_FOLDER / "establecimiento.xls"

    nuevo = RAW_FOLDER / f"{nombre_departamento}.xls"

    if nuevo.exists():
        nuevo.unlink()

    archivo.rename(nuevo)

    return nuevo

In [17]:
options = webdriver.ChromeOptions()

prefs = {
    "download.default_directory": str(RAW_FOLDER.resolve()),
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True
}

options.add_experimental_option("prefs", prefs)

options.add_argument("--start-maximized")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

wait = WebDriverWait(driver,20)

driver.get(URL)

In [18]:
for codigo, nombre in departamentos.items():

    print("="*60)
    print(f"Descargando {nombre}")
    print("="*60)

    # Departamento

    Select(
        esperar_elemento(
            driver,
            By.ID,
            "_ctl0_ContentPlaceHolder1_cmbDepartamento"
        )
    ).select_by_value(codigo)

    # Nivel

    Select(
        esperar_elemento(
            driver,
            By.ID,
            "_ctl0_ContentPlaceHolder1_cmbNivel"
        )
    ).select_by_value("46")

    # Buscar

    esperar_click(
        driver,
        By.ID,
        "_ctl0_ContentPlaceHolder1_IbtnConsultar"
    ).click()

    # Esperar que aparezca el botón Exportar

    esperar_click(
        driver,
        By.ID,
        "_ctl0_ContentPlaceHolder1_btnExportar",
        timeout=30
    )

    # Exportar

    driver.find_element(
        By.ID,
        "_ctl0_ContentPlaceHolder1_btnExportar"
    ).click()

    # Esperar descarga

    esperar_descarga(RAW_FOLDER, DOWNLOAD_TIMEOUT)

    # Renombrar

    renombrar_excel(nombre)

    print(f"{nombre} descargado correctamente.\n")

Descargando Alta_Verapaz
Alta_Verapaz descargado correctamente.

Descargando Baja_Verapaz
Baja_Verapaz descargado correctamente.

Descargando Chimaltenango
Chimaltenango descargado correctamente.

Descargando Chiquimula
Chiquimula descargado correctamente.

Descargando Ciudad_Capital
Ciudad_Capital descargado correctamente.

Descargando El_Progreso
El_Progreso descargado correctamente.

Descargando Escuintla
Escuintla descargado correctamente.

Descargando Guatemala
Guatemala descargado correctamente.

Descargando Huehuetenango
Huehuetenango descargado correctamente.

Descargando Izabal
Izabal descargado correctamente.

Descargando Jalapa
Jalapa descargado correctamente.

Descargando Jutiapa
Jutiapa descargado correctamente.

Descargando Peten
Peten descargado correctamente.

Descargando Quetzaltenango
Quetzaltenango descargado correctamente.

Descargando Quiche
Quiche descargado correctamente.

Descargando Retalhuleu
Retalhuleu descargado correctamente.

Descargando Sacatepequez
Sacat

In [19]:
print("="*50)
print("Descarga completada.")
print(f"Departamentos descargados: {len(departamentos)}")
print(f"Archivos guardados en: {RAW_FOLDER.resolve()}")
print("="*50)

Descarga completada.
Departamentos descargados: 23
Archivos guardados en: C:\Users\Emadlg\Desktop\UVG\Cuarto Año\Segundo Ciclo\Data Science\Data_Science\Proyecto1\data\raw


# Parte 2 - Conversión y Unión de los Datos

En esta sección se leerán todos los archivos Excel descargados automáticamente,
se convertirán a DataFrames de pandas y posteriormente se unirán en un único
conjunto de datos que servirá como base para el proceso de limpieza.

In [25]:
%pip install lxml html5lib beautifulsoup4

     ---------------------------------------- 4.0/4.0 MB 9.5 MB/s eta 0:00:00
     ---------------------------------------- 112.2/112.2 kB ? eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
from glob import glob

In [27]:
# Buscar todos los Excel descargados

excel_files = sorted(glob(str(RAW_FOLDER / "*.xls")))

print(f"Se encontraron {len(excel_files)} archivos.")

Se encontraron 23 archivos.


In [32]:
dataframes = []

for archivo in excel_files:

    print(f"Leyendo {Path(archivo).name}")

    tablas = pd.read_html(archivo)

    tabla_correcta = None

    for tabla in tablas:

        # Buscar si la PRIMERA FILA contiene "ESTABLECIMIENTO"
        primera_fila = tabla.iloc[0].astype(str).str.upper().tolist()

        if "ESTABLECIMIENTO" in primera_fila:

            tabla_correcta = tabla.copy()
            break

    if tabla_correcta is None:
        print(f"No se encontró la tabla de datos en {archivo}")
        continue

    # La primera fila realmente son los encabezados
    tabla_correcta.columns = tabla_correcta.iloc[0]

    # Eliminar esa fila del contenido
    tabla_correcta = tabla_correcta.iloc[1:].reset_index(drop=True)

    tabla_correcta["ARCHIVO_ORIGEN"] = Path(archivo).stem

    dataframes.append(tabla_correcta)

Leyendo Alta_Verapaz.xls
Leyendo Baja_Verapaz.xls
Leyendo Chimaltenango.xls
Leyendo Chiquimula.xls
Leyendo Ciudad_Capital.xls
Leyendo El_Progreso.xls
Leyendo Escuintla.xls
Leyendo Guatemala.xls
Leyendo Huehuetenango.xls
Leyendo Izabal.xls
Leyendo Jalapa.xls
Leyendo Jutiapa.xls
Leyendo Peten.xls
Leyendo Quetzaltenango.xls
Leyendo Quiche.xls
Leyendo Retalhuleu.xls
Leyendo Sacatepequez.xls
Leyendo San_Marcos.xls
Leyendo Santa_Rosa.xls
Leyendo Solola.xls
Leyendo Suchitepequez.xls
Leyendo Totonicapan.xls
Leyendo Zacapa.xls


In [33]:
len(dataframes)

23

In [34]:
df_raw = pd.concat(dataframes, ignore_index=True)

df_raw.head()

,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL,ARCHIVO_ORIGEN
0,16-01-0137-46,16-006,ALTA VERAPAZ,COBAN,INSTITUTO MIXTO NOCTURNO FRANCISCO MARROQUIN,6A. AVENIDA 1-15 ZONA 4,NaN,JORGE EDUARDO PAQUE LAZARO,--,DIVERSIFICADO,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,NOCTURNA,DIARIO(REGULAR),ALTA VERAPAZ,Alta_Verapaz
1,16-01-0138-46,16-01-0926,ALTA VERAPAZ,COBAN,COLEGIO COBAN,KM.2 SALIDA A SAN JUAN CHAMELCO ZONA 8,77945104,JOSE ARTURO CHOC CHEN,GUSTAVO ADOLFO SIERRA POP,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ,Alta_Verapaz
2,16-01-0139-46,16-01-0927,ALTA VERAPAZ,COBAN,COLEGIO PARTICULAR MIXTO VERAPAZ,KM 209.5 ENTRADA A LA CIUDAD,58015763,ARMANDO AUDILIO POP CUCUL,GILMA DOLORES GUAY PAZ DE LEAL,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ,Alta_Verapaz
3,16-01-0140-46,16-01-0926,ALTA VERAPAZ,COBAN,"COLEGIO ""LA INMACULADA""",7A. AVENIDA 11-109 ZONA 6,78232301,JOSE ARTURO CHOC CHEN,VIRGINIA SOLANO SERRANO,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ,Alta_Verapaz
4,16-01-0141-46,16-01-0924,ALTA VERAPAZ,COBAN,ESCUELA NACIONAL DE CIENCIAS COMERCIALES,2A CALLE 11-10 ZONA 2,40389968,MOISÉS ADRIÁN LÓPEZ PÉREZ,ESTELA DEL CARMEN QUIM ROSALES,DIVERSIFICADO,OFICIAL,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ,Alta_Verapaz


In [37]:
print(f"Departamentos cargados: {len(dataframes)}")

Departamentos cargados: 23


In [38]:
df_raw = pd.concat(
    dataframes,
    ignore_index=True
)

In [39]:
print(df_raw.shape)

(11891, 18)


In [40]:
csv_path = CSV_FOLDER / "establecimientos_crudos.csv"

df_raw.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

print("CSV generado correctamente.")
print(csv_path)

CSV generado correctamente.
data\csv\establecimientos_crudos.csv


In [41]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 11891 entries, 0 to 11890
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   CODIGO           11868 non-null  str  
 1   DISTRITO         11336 non-null  str  
 2   DEPARTAMENTO     11868 non-null  str  
 3   MUNICIPIO        11868 non-null  str  
 4   ESTABLECIMIENTO  11863 non-null  str  
 5   DIRECCION        11792 non-null  str  
 6   TELEFONO         10922 non-null  str  
 7   SUPERVISOR       11333 non-null  str  
 8   DIRECTOR         10137 non-null  str  
 9   NIVEL            11868 non-null  str  
 10  SECTOR           11868 non-null  str  
 11  AREA             11868 non-null  str  
 12  STATUS           11868 non-null  str  
 13  MODALIDAD        11868 non-null  str  
 14  JORNADA          11868 non-null  str  
 15  PLAN             11868 non-null  str  
 16  DEPARTAMENTAL    11868 non-null  str  
 17  ARCHIVO_ORIGEN   11891 non-null  str  
dtypes: str(18)
memory

In [42]:
print("="*50)

print(f"Registros : {df_raw.shape[0]}")
print(f"Variables : {df_raw.shape[1]}")

print("="*50)

Registros : 11891
Variables : 18


# Parte 3 - Diagnóstico del Conjunto de Datos

En esta sección se realiza un análisis exploratorio del conjunto de datos obtenido, con el objetivo de identificar problemas de calidad antes de iniciar el proceso de limpieza.

Se evaluarán:

- Número de registros y variables.
- Tipos de datos.
- Valores faltantes.
- Valores únicos.
- Registros duplicados.
- Posibles inconsistencias.

In [43]:
print("="*60)
print("DIMENSIONES DEL DATASET")
print("="*60)

print(f"Registros : {df_raw.shape[0]}")
print(f"Variables : {df_raw.shape[1]}")

DIMENSIONES DEL DATASET
Registros : 11891
Variables : 18


In [44]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 11891 entries, 0 to 11890
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   CODIGO           11868 non-null  str  
 1   DISTRITO         11336 non-null  str  
 2   DEPARTAMENTO     11868 non-null  str  
 3   MUNICIPIO        11868 non-null  str  
 4   ESTABLECIMIENTO  11863 non-null  str  
 5   DIRECCION        11792 non-null  str  
 6   TELEFONO         10922 non-null  str  
 7   SUPERVISOR       11333 non-null  str  
 8   DIRECTOR         10137 non-null  str  
 9   NIVEL            11868 non-null  str  
 10  SECTOR           11868 non-null  str  
 11  AREA             11868 non-null  str  
 12  STATUS           11868 non-null  str  
 13  MODALIDAD        11868 non-null  str  
 14  JORNADA          11868 non-null  str  
 15  PLAN             11868 non-null  str  
 16  DEPARTAMENTAL    11868 non-null  str  
 17  ARCHIVO_ORIGEN   11891 non-null  str  
dtypes: str(18)
memory

In [45]:
tipos = pd.DataFrame({
    "Variable": df_raw.columns,
    "Tipo de dato": df_raw.dtypes.astype(str)
})

tipos

,Variable,Tipo de dato
0,,
CODIGO,CODIGO,str
DISTRITO,DISTRITO,str
DEPARTAMENTO,DEPARTAMENTO,str
MUNICIPIO,MUNICIPIO,str
ESTABLECIMIENTO,ESTABLECIMIENTO,str
DIRECCION,DIRECCION,str
TELEFONO,TELEFONO,str
SUPERVISOR,SUPERVISOR,str
DIRECTOR,DIRECTOR,str


In [46]:
faltantes = pd.DataFrame({
    "Variable": df_raw.columns,
    "Valores faltantes": df_raw.isna().sum(),
    "Porcentaje": (
        df_raw.isna().sum() / len(df_raw) * 100
    ).round(2)
})

faltantes.sort_values(
    "Valores faltantes",
    ascending=False
)

,Variable,Valores faltantes,Porcentaje
0,,,
DIRECTOR,DIRECTOR,1754,14.75
TELEFONO,TELEFONO,969,8.15
SUPERVISOR,SUPERVISOR,558,4.69
DISTRITO,DISTRITO,555,4.67
DIRECCION,DIRECCION,99,0.83
ESTABLECIMIENTO,ESTABLECIMIENTO,28,0.24
CODIGO,CODIGO,23,0.19
MUNICIPIO,MUNICIPIO,23,0.19
DEPARTAMENTO,DEPARTAMENTO,23,0.19


In [47]:
unicos = pd.DataFrame({
    "Variable": df_raw.columns,
    "Valores únicos": df_raw.nunique()
})

unicos

,Variable,Valores únicos
0,,
CODIGO,CODIGO,11868
DISTRITO,DISTRITO,1681
DEPARTAMENTO,DEPARTAMENTO,23
MUNICIPIO,MUNICIPIO,352
ESTABLECIMIENTO,ESTABLECIMIENTO,6309
DIRECCION,DIRECCION,7438
TELEFONO,TELEFONO,6573
SUPERVISOR,SUPERVISOR,1283
DIRECTOR,DIRECTOR,5521


In [48]:
duplicados = df_raw.duplicated().sum()

print(f"Duplicados exactos: {duplicados}")

Duplicados exactos: 0


In [49]:
texto = df_raw.select_dtypes(include="object").columns

espacios = {}

for columna in texto:

    espacios[columna] = (
        df_raw[columna]
        .fillna("")
        .astype(str)
        .str.startswith(" ")
        .sum()
        +
        df_raw[columna]
        .fillna("")
        .astype(str)
        .str.endswith(" ")
        .sum()
    )

pd.DataFrame.from_dict(
    espacios,
    orient="index",
    columns=["Registros con espacios"]
)

C:\Users\Emadlg\AppData\Local\Temp\ipykernel_24288\3430575785.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  texto = df_raw.select_dtypes(include="object").columns


,Registros con espacios
CODIGO,0
DISTRITO,0
DEPARTAMENTO,0
MUNICIPIO,0
ESTABLECIMIENTO,0
DIRECCION,0
TELEFONO,0
SUPERVISOR,0
DIRECTOR,0
NIVEL,0


In [50]:
vacias = {}

for columna in texto:

    vacias[columna] = (
        df_raw[columna]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

pd.DataFrame.from_dict(
    vacias,
    orient="index",
    columns=["Cadenas vacías"]
)

,Cadenas vacías
CODIGO,23
DISTRITO,555
DEPARTAMENTO,23
MUNICIPIO,23
ESTABLECIMIENTO,28
DIRECCION,99
TELEFONO,969
SUPERVISOR,558
DIRECTOR,1754
NIVEL,23


In [51]:
resumen = pd.DataFrame({

    "Métrica":[

        "Registros",
        "Variables",
        "Duplicados exactos",
        "Variables con NA",
        "Valores faltantes"

    ],

    "Valor":[

        len(df_raw),
        df_raw.shape[1],
        df_raw.duplicated().sum(),
        (df_raw.isna().sum()>0).sum(),
        df_raw.isna().sum().sum()

    ]

})

resumen

,Métrica,Valor
0,Registros,11891
1,Variables,18
2,Duplicados exactos,0
3,Variables con NA,17
4,Valores faltantes,4216
